# Phase 2: Preprocessing and Tokenization
In this notebook, we will clean the SMILES dataset, build a vocabulary, and convert the text into numerical tensors that the VAE can understand.

In [1]:
import pandas as pd
import numpy as np
import json
from rdkit import Chem
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

## 1. Load and Clean the Data
We will load the raw dataset and remove any SMILES strings that are too long (which slows down training).

In [2]:
data_path = '../data/raw/250k_rndm_zinc_drugs_clean_3.csv'
df = pd.read_csv(data_path)

smiles_col = 'smiles' if 'smiles' in df.columns else df.columns[0]

# Remove duplicates and missing values
df = df.dropna(subset=[smiles_col])
df = df.drop_duplicates(subset=[smiles_col])

# Calculate lengths
df['length'] = df[smiles_col].apply(lambda x: len(str(x)))

# Filter out very long molecules (e.g., length > 120)
MAX_SMILES_LEN = 120
df_clean = df[df['length'] <= MAX_SMILES_LEN].copy()

print(f"Original dataset size: {len(df)}")
print(f"Cleaned dataset size: {len(df_clean)}")

Original dataset size: 249455
Cleaned dataset size: 249455


## 2. Build the Vocabulary
Neural networks only understand numbers. We need to create a "dictionary" that maps every character in the SMILES strings to a unique integer.

In [3]:
# Extract all unique characters from the dataset
charset = set()
for s in df_clean[smiles_col]:
    charset.update(list(str(s)))

charset = sorted(list(charset))

# Add Special Tokens
# <PAD> to make sequences the same length
# <SOS> Start of sequence
# <EOS> End of sequence
vocab = ['<PAD>', '<SOS>', '<EOS>'] + charset

# Create mapping dictionaries
char_to_int = {c: i for i, c in enumerate(vocab)}
int_to_char = {i: c for i, c in enumerate(vocab)}

print(f"Vocabulary size: {len(vocab)} characters")
print("Vocabulary:", vocab)

Vocabulary size: 38 characters
Vocabulary: ['<PAD>', '<SOS>', '<EOS>', '\n', '#', '(', ')', '+', '-', '/', '1', '2', '3', '4', '5', '6', '7', '8', '=', '@', 'B', 'C', 'F', 'H', 'I', 'N', 'O', 'P', 'S', '[', '\\', ']', 'c', 'l', 'n', 'o', 'r', 's']


## 3. Tokenize and Pad
Now we convert every SMILES string into a list of integers, adding `<SOS>` at the beginning and `<EOS>` at the end.

In [4]:
def encode_smiles(smiles, max_length, char_to_int):
    # Start token
    encoded = [char_to_int['<SOS>']]
    
    # Characters
    for char in str(smiles):
        encoded.append(char_to_int[char])
        
    # End token
    encoded.append(char_to_int['<EOS>'])
    
    # Padding
    # The total target length is max_length + 2 (for SOS and EOS)
    pad_length = (max_length + 2) - len(encoded)
    if pad_length > 0:
        encoded.extend([char_to_int['<PAD>']] * pad_length)
        
    return encoded

# We will use the MAX_SMILES_LEN we defined earlier
encoded_data = []
for smiles in df_clean[smiles_col]:
    encoded_data.append(encode_smiles(smiles, MAX_SMILES_LEN, char_to_int))

encoded_data = np.array(encoded_data)
print(f"Encoded data shape: {encoded_data.shape}")

# Let's look at one example
print("\nOriginal SMILES:", df_clean[smiles_col].iloc[0])
print("Encoded:", encoded_data[0])

Encoded data shape: (249455, 122)

Original SMILES: CC(C)(C)c1ccc2occ(CC(=O)Nc3ccccc3F)c2c1

Encoded: [ 1 21 21  5 21  6  5 21  6 32 10 32 32 32 11 35 32 32  5 21 21  5 18 26
  6 25 32 12 32 32 32 32 32 12 22  6 32 11 32 10  3  2  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0]


## 4. Train/Test Split and Save
We split the data so we can validate the model, and then save all processed files so our Model Training notebook can use them.

In [5]:
# Split 90% for training, 10% for validation/testing
X_train, X_test = train_test_split(encoded_data, test_size=0.1, random_state=42)

print(f"Train size: {X_train.shape}")
print(f"Test size: {X_test.shape}")

# Save Numpy arrays
np.save('../data/processed/X_train.npy', X_train)
np.save('../data/processed/X_test.npy', X_test)

# Save Vocabulary
vocab_data = {
    'char_to_int': char_to_int,
    'int_to_char': int_to_char,
    'max_len': MAX_SMILES_LEN
}
with open('../data/processed/vocab.json', 'w') as f:
    json.dump(vocab_data, f)

print("\nPreprocessing Complete! Files saved to data/processed/")

Train size: (224509, 122)
Test size: (24946, 122)

Preprocessing Complete! Files saved to data/processed/
